![Platform+](images/banner.png)

# LAB 5: APPLY AI TO TEXT
_Hands-on Snowflake for data practitioners_

> **Persona track: Data Analyst** — same lab, your angle.

👉 **In this lesson, we'll turn raw customer comments into actionable intelligence using Cortex AI functions.**

📚 **By the end you'll be able to:**
- Score sentiment on a text column without training a model
- Classify free text into custom categories
- Roll up many comments into concise executive summaries
- Reason about AI token cost before scaling

**Time:** ~30 min · **Lab interface:** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)

## Set Up Your Workspace
First, set up your **personal workspace**. The cell below switches to your own role, creates and uses your own warehouse, and creates your own database for what you'll build in this lab. Every object is named after your username, so everyone in the account stays out of each other's way.

In [ ]:
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Your personal database + the schema(s) this lab writes to
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.SANDBOX;
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

> 📌 **Why you can call these AI functions:** access to Cortex AI functions is granted through the [`SNOWFLAKE.CORTEX_USER`](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access) **database role**, already granted to your role during setup.
>
> A **database role** is a role whose privileges are scoped to a single database — here, the built-in `SNOWFLAKE` database. Unlike an account role, a database role can't be activated directly in a session; it's *granted to* your account role (`{{my_role}}`), which then inherits its privileges. It's Snowflake's standard way to package up "the privileges needed to use feature X" and hand them to whoever should have them. More on [database roles](https://docs.snowflake.com/en/user-guide/security-access-control-overview). Everything you build in this lab still lands in your own database.

## Snowflake AI Functions — the big picture
Snowflake gives you a whole **family of AI functions** (part of *Cortex AISQL*) that run large language models for you, right inside the database. Here's the magic: **you don't change how you work.** No model to train, no GPUs to provision, no data to ship out to an external API. You call them like any other function — in **SQL or Python** — on data that never leaves Snowflake. That's what makes them so powerful: enterprise-grade AI becomes just another function in your query.

There are many of these functions (sentiment, classification, extraction, translation, summarization, embeddings, and more). In this lab you'll use four as a Data Analyst:

| Function | What it does |
|---|---|
| `AI_SENTIMENT` | Scores the sentiment of text (positive / negative / neutral / mixed) |
| `AI_CLASSIFY` | Sorts text into categories you define |
| `AI_AGG` | Reads many rows and returns one AI summary per group |
| `AI_COUNT_TOKENS` | Estimates the token cost of an AI call before you run it at scale |

👉 Browse the full list any time in the [Cortex AI Functions docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql).

## 1 · Know Your Data
Before applying AI, understand what you're working with. The `FEEDBACK` table holds free-text comments from customers across all roastery stores, along with a numeric 1–5 `SCORE` each customer gave. Think of `SCORE` as a star rating: **1 = very unhappy, 5 = delighted** (3 is lukewarm). It's a quick pulse — but on its own it can't tell you *what* drove the rating. That "why" lives in the free-text comment, which is exactly what AI lets us read at scale.

Let's peek at the data.

In [ ]:
-- COMMENT is a reserved word in Snowflake, so we alias it to CUSTOMER_COMMENT for a clean result
SELECT feedback_id, store_id, customer_id, score,
       comment AS customer_comment
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
LIMIT 10;

> 🔎 **Review the output.** The numeric `score` tells you *how happy* someone is, but the `customer_comment` tells you *why*. AI functions let us read the 'why' at scale.

## 2 · Score Sentiment With AI_SENTIMENT
[`AI_SENTIMENT`](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment) reads a text column and returns a structured object. The overall sentiment is a **category** — `positive`, `negative`, `neutral`, or `mixed` — at `:categories[0]:sentiment`. No model training, no data movement — just SQL.

> 📎 **A quick note on the syntax.** AI_SENTIMENT hands back a structured (JSON-like) object, not a plain word. The `:categories[0]:sentiment` part just *navigates into* that object to reach the value you want, and `::STRING` casts it to text. Don't get bogged down in it here — just recognize that this path-and-cast pattern (`:key` to step into a field, `[0]` to index an array, `::STRING` to cast) is how you read a specific field out of any AI function's structured result.

👉 Score a sample and compare the AI category to the customer's own rating.

In [ ]:
SELECT feedback_id,
       score AS customer_rating,
       AI_SENTIMENT(comment):categories[0]:sentiment::STRING AS ai_sentiment,
       LEFT(comment, 80) AS comment_preview
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE comment IS NOT NULL
LIMIT 15;

> 🔎 **Review the output.** Does the AI category track with the 1–5 customer rating? Usually yes — but sometimes a 3-star review has very negative *text*, which the AI catches. That's the value: nuance the number misses.

> 📌 **Two flavors of sentiment scoring.** Snowflake offers both `AI_SENTIMENT` (returns a categorical object: positive/negative/neutral/mixed) and `SNOWFLAKE.CORTEX.SENTIMENT` (returns a numeric float from -1 to +1). Use the categorical version when you need labels for filtering or grouping. Use the numeric version when you need to `AVG()` or chart a score. Both are valid — pick the one that fits your downstream use.

### Visualize: average sentiment by store. 📊
We map each category to a number (positive = +1, mixed = 0, negative = -1) and average per store. This is a **Python** cell: `session.sql(...).to_pandas()` runs the query and pulls the result into a pandas DataFrame, then **matplotlib** draws the bars. Taller bar = happier customers.

> 💡 **Why matplotlib?** It's just *one* of many Python charting libraries (Altair, Plotly, and seaborn are others) — we use it here because it's everywhere and takes only a few lines once your data is in a DataFrame. Because you're already in Python, wiring up a chart is easy — and if you're unsure of the syntax, that's exactly the kind of thing to hand to CoCo: *"chart this DataFrame as a bar chart with matplotlib."*

In [ ]:
import matplotlib.pyplot as plt
df = session.sql("""
    SELECT s.store_name,
           ROUND(AVG(CASE AI_SENTIMENT(f.comment):categories[0]:sentiment::STRING
               WHEN 'positive' THEN 1 WHEN 'mixed' THEN 0 WHEN 'negative' THEN -1
           END), 3) AS avg_sentiment,
           COUNT(*) AS review_count
    FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK f
    JOIN PLATPLUS_DB.ROASTERY_SALES.STORES s ON s.store_id = f.store_id
    WHERE f.comment IS NOT NULL
    GROUP BY s.store_name
    ORDER BY avg_sentiment
""").to_pandas()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df['STORE_NAME'], df['AVG_SENTIMENT'], color='#29B5E8')
ax.set_xlabel('Store')
ax.set_ylabel('Avg sentiment (+1 positive / -1 negative)')
ax.set_title('Average AI Sentiment by Store')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
df

## 3 · Classify Issues With AI_CLASSIFY
Knowing sentiment is negative isn't enough — store managers need to know **what went wrong**.

[`AI_CLASSIFY`](https://docs.snowflake.com/en/sql-reference/functions/ai_classify) maps free text into categories *you* define. You hand it the text plus a list of candidate labels, and it returns an object — the chosen label sits at `:labels[0]` (the same navigate-and-cast pattern as before).

Let's classify negative comments into actionable buckets. The query below looks only at comments that AI_SENTIMENT already flagged as `negative` (that's the `WHERE` clause), then asks AI_CLASSIFY to drop each one into a bucket like *Slow Service* or *Product Quality* — turning a pile of complaints into a countable, actionable list.

In [ ]:
SELECT feedback_id,
       AI_CLASSIFY(
           comment,
           ['Slow Service', 'Product Quality', 'Cleanliness',
            'Staff Attitude', 'Pricing', 'Other']
       ):labels[0]::STRING AS issue_category,
       LEFT(comment, 80) AS comment_preview
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE comment IS NOT NULL
  AND AI_SENTIMENT(comment):categories[0]:sentiment::STRING = 'negative'
LIMIT 20;

> 🔎 **Review the output.** Read 3–4 rows and decide: did the AI pick the right category? If not, consider adjusting your category labels — the model takes them literally.

### Which issue categories appear most often?
🧩 **Your turn.** The query below is complete except for one blank on the `GROUP BY` line. To count how many comments land in each bucket, you need to group by the column that holds the category — the alias on the `AI_CLASSIFY` result up in the `SELECT`. Fill in `______`, then run it. Stuck? Reveal the reference solution just below the cell.

In [ ]:
SELECT AI_CLASSIFY(
           comment,
           ['Slow Service', 'Product Quality', 'Cleanliness',
            'Staff Attitude', 'Pricing', 'Other']
       ):labels[0]::STRING AS issue_category,
       COUNT(*) AS issue_count
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE comment IS NOT NULL
  AND AI_SENTIMENT(comment):categories[0]:sentiment::STRING = 'negative'
GROUP BY ______   -- 🧩 fill in the column to group by
ORDER BY issue_count DESC;

> <details><summary>🧩 Stuck? Reference solution</summary>
>
>     GROUP BY issue_category
>
> `issue_category` is the alias we gave the `AI_CLASSIFY` result in the `SELECT` list — it's the column holding each comment's bucket.
> </details>

## 4 · Summarize per Store With AI_AGG
Now give each store manager a 2-sentence executive summary of their feedback themes.

[`AI_AGG`](https://docs.snowflake.com/en/sql-reference/functions/ai_agg) is an **aggregate** function — it reads many rows and returns one AI-generated summary per group. That's the key to our goal: because it aggregates like `SUM` or `COUNT` (but with language instead of math), pairing it with `GROUP BY store_name` collapses every comment for a store into a *single* summary row. One store → one manager → one summary. The second argument is a plain-English instruction telling the model what kind of summary you want.

In [ ]:
SELECT s.store_name,
       AI_AGG(
           f.comment,
           'Summarize the main themes in 2 sentences. Focus on what customers love and what needs improvement.'
       ) AS feedback_summary
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK f
JOIN PLATPLUS_DB.ROASTERY_SALES.STORES s ON s.store_id = f.store_id
WHERE f.comment IS NOT NULL
GROUP BY s.store_name;

> 🔎 **Review the output.** Pick one store and scan its raw comments. Does the AI summary capture the themes fairly? This is how you build trust in AI outputs before sharing with stakeholders.

## 5 · Save Your Results
Persist the sentiment-scored data into your own database so you can build dashboards without re-running AI each time. This is a key analyst pattern: **compute once, query many times**.

> 💡 **Table or view?** We use `CREATE TABLE … AS` (a *CTAS*) here on purpose. A **view** would re-run `AI_SENTIMENT` and `AI_CLASSIFY` *every time anyone queried it* — paying the AI token cost again and again. A **table** runs the AI once and freezes the results, so downstream dashboards are fast and cost nothing extra. The trade-off: a table is a **point-in-time copy** — if new feedback lands, you'd re-run this to refresh it (not a concern for this lab). Pulling a transformed copy out of the raw source like this — enriching it, then querying your copy — is a completely normal, safe analyst pattern: you're never touching the shared source data.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.SANDBOX.FEEDBACK_SCORED AS
SELECT f.feedback_id, f.store_id, s.store_name,
       f.score AS customer_rating,
       AI_SENTIMENT(f.comment):categories[0]:sentiment::STRING AS ai_sentiment,
       AI_CLASSIFY(
           f.comment,
           ['Slow Service', 'Product Quality', 'Cleanliness',
            'Staff Attitude', 'Pricing', 'Positive']
       ):labels[0]::STRING AS issue_category,
       f.comment
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK f
JOIN PLATPLUS_DB.ROASTERY_SALES.STORES s ON s.store_id = f.store_id
WHERE f.comment IS NOT NULL;

In [ ]:
-- Quick check: counts and average sentiment per issue category
SELECT issue_category,
       COUNT(*) AS n,
       ROUND(AVG(CASE ai_sentiment
           WHEN 'positive' THEN 1 WHEN 'mixed' THEN 0 WHEN 'negative' THEN -1
       END), 2) AS avg_sent
FROM {{my_db}}.SANDBOX.FEEDBACK_SCORED
GROUP BY issue_category
ORDER BY n DESC;

## 6 · Cost Awareness
AI functions bill per **token** processed. A **token** is a chunk of text the model reads — very roughly **~4 characters of English** — so a short comment might be 15–30 tokens. Every AI call turns your text into tokens; Snowflake meters the tokens consumed and converts them to **credits** at a per-model rate, and credits are Snowflake's billing unit (the same credits you costed out in Lab 4). More text → more tokens → more credits, so cost scales with how much text you feed the model.

The query below uses [`AI_COUNT_TOKENS`](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens) to **estimate cost before you commit to it**. It counts the tokens `AI_SENTIMENT` *would* process across the entire `FEEDBACK` table — total and per-row average — without actually calling the model. We run it on the full `FEEDBACK` source precisely because that answers the real-world question: *"what would it cost to score everything?"* `AI_COUNT_TOKENS` itself only incurs a little compute (not AI tokens), so it's a safe way to look before you leap.

In [ ]:
-- Estimate token cost for AI_SENTIMENT over the feedback table
SELECT COUNT(*) AS total_rows,
       SUM(AI_COUNT_TOKENS('ai_sentiment', comment)) AS total_tokens,
       ROUND(AVG(AI_COUNT_TOKENS('ai_sentiment', comment)), 1) AS avg_tokens
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE comment IS NOT NULL;

> 📌 **Note:** Always test on a `LIMIT` sample first, then estimate full-table cost before running at scale. Pre-filter rows (e.g., only this month's feedback) to control spend.

---
## 🤖 CoCo Assist
You've now met four AI functions. In the real world you won't always know *which* one fits a new question — and that's fine. This is exactly where CoCo earns its keep: describe your goal in plain English, let it point you to the right function, then help you write the SQL.

Your manager asks: *"Which products get the most complaints?"*

> 📌 **The catch.** There's no clean join for this. `FEEDBACK` has no `PRODUCT_ID` or `ORDER_ID` — the product a customer is unhappy about is named *inside the free-text* `COMMENT`. So this isn't a join problem; it's a job for an AI function. But *which* one? Don't guess — ask CoCo.

**Step 1 — ask CoCo which function fits:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">I have a table FEEDBACK in @PLATPLUS_DB.ROASTERY_SALES with a free-text COMMENT column but no product_id — the product is only mentioned inside the comment text. I want to know which product each complaint is about. Which Snowflake Cortex AI function should I use, and why? Stick to functions I can call directly in SQL.</pre>

> 🔎 **Read CoCo's recommendation.** It should steer you to **`AI_EXTRACT`** — which pulls a named value *out of* text. Notice this is a function you haven't used in this lab: that's the point. You don't need to memorize the catalog; CoCo knows it and maps your goal to the right tool.
>
> **Why AI_EXTRACT and not AI_CLASSIFY here?** `AI_CLASSIFY` *forces* every row into one of the labels you give it — so a complaint that names no product still gets jammed into some bucket (misleading). `AI_EXTRACT` pulls the product when it's actually mentioned and returns **null** when it isn't — which is exactly what you want, since many complaints are about service, not a product.

**Step 2 — ask CoCo to write it:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Using @PLATPLUS_DB.ROASTERY_SALES, write a query that uses AI_EXTRACT to pull the product name mentioned in each negative-sentiment comment in FEEDBACK, returning null when no product is named. Then count how many complaints mention each product (most first), and include a count of comments that named no product at all.</pre>

Review what CoCo suggests, paste it below, and run it.

In [ ]:
-- Paste CoCo's Step 2 query on the blank line below, then run it


### Iterate — sharpen the extraction 💡
Run your query and scan the results. Lots of nulls, or the same product spelled inconsistently? Extraction quality depends on how clearly you specify the field you want. Rather than hand-tuning, ask CoCo:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">My AI_EXTRACT for product name returns many nulls or inconsistent spellings from short customer comments. Suggest a clearer field name or a short instruction to improve extraction accuracy, and how to normalize the extracted values against our PRODUCTS list.</pre>

### More ways to lean on CoCo with AI functions 🤖
CoCo isn't just for writing queries. A few prompts worth keeping in your back pocket:
- **Decode a result:** *"Explain this AI_SENTIMENT object and how to pull the confidence score, not just the label."*
- **Estimate cost:** *"Estimate the cost of running AI_SENTIMENT on 2 million rows and suggest ways to reduce it."*
- **Compare functions:** *"AI_CLASSIFY vs AI_EXTRACT — when do I sort text into fixed buckets vs pull a value out of it?"*

## Check Your Work
Run the next cell to **check your work**. It queries your own database to confirm you created the objects and rows this lab asked for, and prints a ✅ / ⛔ result for each — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
checks = [
    ('Your SANDBOX.FEEDBACK_SCORED has AI-scored rows', f'SELECT COUNT(*) FROM {my_db}.SANDBOX.FEEDBACK_SCORED', 1),
]
passed = 0
for label, sql, need in checks:
    try:
        n = session.sql(sql).collect()[0][0]
        ok = n >= need
    except Exception:
        n, ok = None, False
    passed += 1 if ok else 0
    icon = '✅' if ok else '⛔'
    detail = f'(found {n})' if n is not None else '(not found)'
    print(f'{icon}  {label}  {detail}')
print('-' * 52)
if passed == len(checks):
    print(f'🎉 ALL {len(checks)} CHECKS PASSED — lab complete! Nice work.')
else:
    print(f'⚠️  {passed}/{len(checks)} passed — re-run the cells above for any ⛔ item, then run this cell again.')

### Knowledge check 🧠
Check your understanding with interactive quiz questions below. Each quiz cell has a widget that presents a multiple‑choice question about Snowflake functionality.

**Instructions:**
1. Hover your cursor over the notebook cell to reveal additional controls.
1. Click the ▶️ **Play button** on the right side of each quiz cell to run it.
1. Select your answer from the options provided.
1. Review the feedback before moving on.

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q3")

### Key takeaways. ❄️
- AI_SENTIMENT returns a sentiment category (positive / negative / neutral / mixed) at :categories[0]:sentiment — no training required
- AI_CLASSIFY maps free text to your custom categories — read the chosen label at :labels[0]
- AI_AGG summarizes many rows into one readable summary per group
- Always test on a sample and estimate token cost before scaling
- Pre-compute and persist AI results (CTAS) — dashboards don't re-run expensive functions

**The Other Personas:**
- The **Data Engineer** notebook builds an AI enrichment pipeline — standardize, extract, translate, embed — then operationalizes it with a Dynamic Table
- The **Platform Admin** notebook adds governance: PII redaction, content moderation, cost monitoring, and model access control

> :bulb: Each persona lab is independent — you can run them in any order.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.